# Exploración de Wine
**Humberto Alejandro Espinola González | A01840066 | TC5061**

Este notebook conserva únicamente exploración de la actividad anterior. El entrenamiento, evaluación y Tracking están en `train.py`. Wine clasifica tres cultivares; no es Wine Quality. La prueba histórica no se utiliza para explorar ni seleccionar configuraciones.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))
from wine_mlops.modeling.train import load_data, split_data
X, y, data_sha256 = load_data()
X_train, X_test, y_train, y_test = split_data(X, y, seed=42)
print("Wine:", X.shape, "| train:", len(X_train), "| test:", len(X_test))
print("SHA-256:", data_sha256)

## Auditoría del esquema
Se conservan los 13 predictores. La clase es categórica aunque se codifique como entero. La carga comprueba etiquetas y duplicados; el esquema y los faltantes se inspeccionan sin aprender transformaciones de prueba.

In [ ]:
display(pd.DataFrame({"tipo": X.dtypes.astype(str), "faltantes": X.isna().sum()}))
display(X_train.head())

## Distribuciones y balance: solo entrenamiento

In [ ]:
display(y_train.value_counts().sort_index().rename("muestras").to_frame())
y_train.value_counts().sort_index().plot.bar(title="Cultivares en entrenamiento", rot=0)
plt.show()
display(X_train.describe().T)
X_train.hist(bins=15, figsize=(15, 12), edgecolor="white")
plt.tight_layout()
plt.show()

In [ ]:
correlation = X_train.corr()
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(correlation, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(X.shape[1]), X.columns, rotation=90)
ax.set_yticks(range(X.shape[1]), X.columns)
fig.colorbar(im, ax=ax)
ax.set_title("Correlación entre predictores: entrenamiento")
plt.tight_layout()
plt.show()

## Decisiones heredadas del baseline
Las escalas distintas justifican StandardScaler. Se conserva la imputación por mediana y todos los extremos: no hay evidencia de errores para eliminarlos. En el script ambas transformaciones se reajustan dentro de cada fold, evitando fuga de información. No se entrena un clasificador en este notebook.

Fuentes: [Wine, UCI](https://archive.ics.uci.edu/dataset/109/wine) y [load_wine, scikit-learn](https://scikit-learn.org/1.7/modules/generated/sklearn.datasets.load_wine.html).